# Chapter 12 - Easy Tasks

In these tasks, you'll practice the basics of fine-tuning generation models with QLoRA.

---

### Setup

In [ ]:
# Install packages (uncomment if on Colab)
# %%capture
# !pip install -q accelerate==0.31.0 peft==0.11.1 bitsandbytes==0.43.1 transformers==4.41.2 trl==0.9.4 sentencepiece==0.2.0

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, TrainingArguments, pipeline
from peft import LoraConfig, prepare_model_for_kbit_training, get_peft_model, AutoPeftModelForCausalLM
from trl import SFTTrainer
from datasets import load_dataset
import warnings
warnings.filterwarnings('ignore')

---

## Task 1: Format Instruction Data

Your first task is to load and format instruction data for fine-tuning.

**Instructions:**
1. Load the UltraChat dataset (use the test_sft split)
2. Select a small subset (1000 examples) for faster training
3. Apply the chat template formatting
4. Examine the formatted output

In [ ]:
# TODO: Load the tokenizer for TinyLlama-1.1B-Chat-v1.0
template_tokenizer = # YOUR CODE HERE

def format_prompt(example):
    """Format the prompt using the chat template."""
    # TODO: Apply the chat template to the messages
    chat = example["messages"]
    prompt = # YOUR CODE HERE (use apply_chat_template)
    return {"text": prompt}

# TODO: Load the dataset
dataset = (
    load_dataset(# YOUR CODE HERE, split=# YOUR CODE HERE)
      .shuffle(seed=42)
      .select(range(# YOUR CODE HERE))  # Select 1000 examples
)

# TODO: Apply the formatting function
dataset = # YOUR CODE HERE

In [ ]:
# Examine a formatted example
print("Dataset size:", len(dataset))
print("\nFirst formatted example:")
print(dataset["text"][0])

### Questions:

**Q1:** What special tokens are used in the chat template?

*Your answer here*

**Q2:** Why do we need to format the data with a chat template instead of just using raw text?

*Your answer here*

---

## Task 2: Configure QLoRA and Train

Now you'll set up QLoRA configuration and train a model.

**Instructions:**
1. Configure 4-bit quantization
2. Load the TinyLlama model with quantization
3. Set up LoRA with rank r=16
4. Train for a short time (50 steps)

In [ ]:
model_name = "TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T"

# TODO: Configure 4-bit quantization
bnb_config = BitsAndBytesConfig(
    load_in_4bit=# YOUR CODE HERE,
    bnb_4bit_quant_type=# YOUR CODE HERE,  # Use "nf4"
    bnb_4bit_compute_dtype=# YOUR CODE HERE,  # Use "float16"
    bnb_4bit_use_double_quant=# YOUR CODE HERE,
)

# TODO: Load the model with quantization
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",
    quantization_config=# YOUR CODE HERE,
)
model.config.use_cache = False
model.config.pretraining_tp = 1

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = "<PAD>"
tokenizer.padding_side = "left"

In [ ]:
# TODO: Configure LoRA with rank r=16
peft_config = LoraConfig(
    lora_alpha=# YOUR CODE HERE,  # Typically 2*r
    lora_dropout=# YOUR CODE HERE,  # Try 0.1
    r=# YOUR CODE HERE,  # Use rank 16
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=['k_proj', 'gate_proj', 'v_proj', 'up_proj', 'q_proj', 'o_proj', 'down_proj']
)

# Prepare model for training
model = prepare_model_for_kbit_training(model)
model = get_peft_model(model, peft_config)

# Check trainable parameters
model.print_trainable_parameters()

In [ ]:
# TODO: Set up training arguments
training_arguments = TrainingArguments(
    output_dir="./tinyllama-easy",
    per_device_train_batch_size=# YOUR CODE HERE,  # Try 2
    gradient_accumulation_steps=# YOUR CODE HERE,  # Try 4
    optim="paged_adamw_32bit",
    learning_rate=# YOUR CODE HERE,  # Try 2e-4
    lr_scheduler_type="cosine",
    max_steps=# YOUR CODE HERE,  # Use 50 steps for quick training
    logging_steps=10,
    fp16=True,
    gradient_checkpointing=True
)

In [ ]:
# TODO: Create and run the trainer
trainer = SFTTrainer(
    model=# YOUR CODE HERE,
    train_dataset=# YOUR CODE HERE,
    dataset_text_field="text",
    tokenizer=# YOUR CODE HERE,
    args=# YOUR CODE HERE,
    max_seq_length=512,
    peft_config=# YOUR CODE HERE,
)

# Train
trainer.train()

# Save
trainer.model.save_pretrained("tinyllama-easy-qlora")

### Questions:

**Q3:** What percentage of parameters are trainable with LoRA compared to the full model?

*Your answer here*

**Q4:** What would happen if you increased the rank (r) from 16 to 64?

*Your answer here*

---

## Task 3: Test the Fine-Tuned Model

Load your fine-tuned model and test it with different prompts.

**Instructions:**
1. Load and merge the LoRA adapters
2. Create a generation pipeline
3. Test with at least 3 different prompts
4. Compare the responses

In [ ]:
# TODO: Load and merge the model
model = AutoPeftModelForCausalLM.from_pretrained(
    # YOUR CODE HERE,  # Path to saved model
    low_cpu_mem_usage=True,
    device_map="auto",
)

merged_model = # YOUR CODE HERE  # Use merge_and_unload()

In [ ]:
# TODO: Create a generation pipeline
pipe = pipeline(
    task=# YOUR CODE HERE,  # "text-generation"
    model=# YOUR CODE HERE,
    tokenizer=# YOUR CODE HERE,
    max_new_tokens=100,
    temperature=0.7,
)

In [ ]:
# Test prompt 1
prompt1 = """<|user|>
What is machine learning?</s>
<|assistant|>
"""

print("Prompt 1: What is machine learning?")
print("="*70)
result = pipe(prompt1)
print(result[0]['generated_text'])
print("\n")

In [ ]:
# TODO: Create and test prompt 2 (ask about a programming concept)
prompt2 = # YOUR CODE HERE

print("Prompt 2: [Your question]")
print("="*70)
# YOUR CODE HERE

In [ ]:
# TODO: Create and test prompt 3 (ask about a scientific topic)
prompt3 = # YOUR CODE HERE

print("Prompt 3: [Your question]")
print("="*70)
# YOUR CODE HERE

### Questions:

**Q5:** How would you describe the quality of the model's responses? Are they coherent and relevant?

*Your answer here*

**Q6:** What differences do you notice between responses when you run the same prompt multiple times?

*Your answer here*

**Q7:** How might training for more steps (e.g., 500 instead of 50) affect the model's performance?

*Your answer here*

---

## Bonus Challenge

Try experimenting with generation parameters:
1. Adjust `temperature` (0.3, 0.7, 1.0) - what changes?
2. Try `do_sample=False` for greedy decoding
3. Adjust `top_p` (nucleus sampling)

Document your findings below:

In [ ]:
# Your experimental code here

*Your observations here*